In [1]:
'''
FP embeddings 2000
LPM embeddings 128
'''

'\nLPM embeddings\n'

In [89]:
import anndata as ad
import pandas as pd
import numpy as np

In [90]:
import perturb_lib as plib

In [91]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [92]:
df_emb = pd.read_pickle('./tahoe_sci_op3_updated.pkl')

In [93]:
df_emb_op3 = df_emb[df_emb['dataset'] == 'op3']

In [94]:
df_emb_op3_add = pd.DataFrame({'perturbagen': ['Belinostat', 'Dabrafenib'],
                                     'smiles': ['O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO', 
                                                'CC(C)(C)c1nc(-c2cccc(NS(=O)(=O)c3c(F)cccc3F)c2F)c(-c2ccnc(N)n2)s1'],
                                     'dataset': ['op3', 'op3'],
                                     'cmap_name': ['belinostat', 'dabrafenib'],
                                     'symbol': ['belinostat-10uM', 'dabrafenib-10uM'],
                                     'code': [8637, 8873],
                                     'symbol_': ['belinostat', 'dabrafenib'],
                                     'original_pert_name': ['Belinostat', 'Dabrafenib']})

In [95]:
df_emb_op3_add['ECFP:2'] = smiles_to_fingerprints(df_emb_op3_add['smiles'])

In [96]:
model = plib.load_trained_model('../perturblib/.plib_cache/results/lincs_paper_lpm/LPM_9bad9756f740b28a/seed_13/model.pt')
emb_list = []
embeddings = model.perturb_embedding_layer.weight.numpy().astype(np.float64)
for c in df_emb_op3_add['code']:
    if pd.isna(c):
        emb_list.append(None)
    else:
        emb_list.append(embeddings[int(c)])

df_emb_op3_add['LPM_emb'] = emb_list

In [97]:
df_emb_op3_add

,perturbagen,smiles,dataset,cmap_name,symbol,code,symbol_,original_pert_name,ECFP:2,LPM_emb
0,Belinostat,O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,op3,belinostat,belinostat-10uM,8637,belinostat,Belinostat,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.0318511500954628, -0.16489124298095703, 0...."
1,Dabrafenib,CC(C)(C)c1nc(-c2cccc(NS(=O)(=O)c3c(F)cccc3F)c2...,op3,dabrafenib,dabrafenib-10uM,8873,dabrafenib,Dabrafenib,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.09784593433141708, -0.4855876863002777, -0...."


In [98]:
df_emb_op3 = pd.concat([df_emb_op3, df_emb_op3_add]).reset_index(drop=True)

In [99]:
de_train = ad.read_h5ad('./data/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
de_test = ad.read_h5ad('./data/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')

In [100]:
id_map = pd.read_csv('./data/benchmark/resources/datasets/neurips-2023-data/id_map.csv')

In [101]:
de_train[de_train.obs['sm_name'].isin(df_emb_op3[~df_emb_op3['code'].isna()]['perturbagen'])].write_h5ad('./data/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad', compression='gzip')
de_test[de_test.obs['sm_name'].isin(df_emb_op3[~df_emb_op3['code'].isna()]['perturbagen'])].write_h5ad('./data/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad', compression='gzip')

In [102]:
id_map[id_map['sm_name'].isin(df_emb_op3[~df_emb_op3['code'].isna()]['perturbagen'])].to_csv('./data/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv', index=False)

In [111]:
df_emb_op3[~df_emb_op3['code'].isna()].reset_index(drop=True).to_pickle("./data/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb.pkl")

In [112]:
pd.read_pickle('./data/benchmark/resources/datasets/neurips-2023-data-subsample/op3_emb.pkl')

,perturbagen,pubchem_cid,smiles,dataset,cmap_name,symbol,code,symbol_,ECFP:2,LPM_emb,original_pert_name
0,TIE2 Kinase Inhibitor,23625762.0,COC1=CC2=C(C=C1)C=C(C=C2)C3=C(NC(=N3)C4=CC=C(C...,op3,BRD-A92800748,BRD-A92800748-10uM,377.0,BRD-A92800748,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.16717687249183655, -0.20145422220230103, 0....",TIE2 Kinase Inhibitor
1,MK-5108,24748204.0,C1CC(CCC1OC2=C(C(=CC=C2)Cl)F)(CC3=NC(=CC=C3)NC...,op3,MK-5108,MK-5108-10uM,7916.0,MK-5108,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.144333153963089, 0.06810502707958221, -0.0...",MK-5108
2,Lapatinib,208908.0,CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3...,op3,lapatinib,lapatinib-10uM,9361.0,lapatinib,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.06751061975955963, -0.12120415270328522, 0...",Lapatinib
3,Atorvastatin,60823.0,CC(C)C1=C(C(=C(N1CCC(CC(CC(=O)O)O)O)C2=CC=C(C=...,op3,atorvastatin,atorvastatin-10uM,8605.0,atorvastatin,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.24961966276168823, -0.1016116589307785, 0....",Atorvastatin
4,Ganetespib (STA-9090),135564985.0,CC(C)C1=C(C=C(C(=C1)C2=NNC(=O)N2C3=CC4=C(C=C3)...,op3,ganetespib,ganetespib-10uM,9186.0,ganetespib,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.11308802664279938, -0.10601439327001572, 0....",Ganetespib (STA-9090)
...,...,...,...,...,...,...,...,...,...,...,...
120,Vanoxerine,3455.0,C1CN(CCN1CCCC2=CC=CC=C2)CCOC(C3=CC=C(C=C3)F)C4...,op3,vanoxerine,vanoxerine-10uM,10195.0,vanoxerine,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.08700791746377945, 0.2101263850927353, -0.0...",Vanoxerine
121,SB525334,9967941.0,CC1=NC(=CC=C1)C2=C(N=C(N2)C(C)(C)C)C3=CC4=NC=C...,op3,SB-525334,SB-525334-10uM,8238.0,SB-525334,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.0031143922824412584, 0.11767911165952682, ...",SB525334
122,HYDROXYUREA,3657.0,C(=O)(N)NO,op3,hydroxyurea,hydroxyurea-10uM,9254.0,hydroxyurea,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.12725546956062317, -0.14163631200790405, -...",HYDROXYUREA
123,Belinostat,NaN,O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,op3,belinostat,belinostat-10uM,8637.0,belinostat,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.0318511500954628, -0.16489124298095703, 0....",Belinostat
